# Implementing a simple AI Agent

In this lab, we will implement a simple AI agent that is able to retrive information from a datasource. This will be done by a mechanic called "Tool calling". 

Let us demystify the concept of an AI Agent. An AI agent is nothig more (**or less**) than a software program build around an LLM that allows the LLM to take autonomous decisions and use tools. Let us break this down:
- A "tool" is an object that allow LLMs to interact with external environments. These tools may be functions made available to LLMs such as a calculator, a search engine, or even another AI—to perform specific tasks it isn't designed to handle on its own. The tools can be executed separately whenever the LLM determines that their use is appropriate.
- A LLM trained to perform supports tool calling (more on this later). Basically, this means that the LLM is able to decide when to use a "tool" .
- Tool calling enables the model to generate a response for a prompt that aligns with a user-defined schema for a function.


that is able to interact with its environment. This interaction can be done in many ways, such as:

## Installation

The first choice to be made is which LLM model to use. Start with small models. The code below has been tested with `llama3.1:8b`  (note: **llama3.1:8b is about 5GB**).

NOTE: Other models may require some adjustments to the prompt. Check the documentation of the model you want to use or test. For instance, if you want to try 'llama3.2:1b' (a smaller model), check the doc in the "Tool Calling (1B/3B)" section, in the [official web site](https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_2/).

In [1]:
model_name = 'llama3.1:8b'

### On Kaggle
Repeat the instructions from the previous part of the lab to install and run Ollama on Kaggle.
The last instruction should print: 

```python
print("Ollama server is running on 127.0.0.1:11438")
```

In [ ]:
# Pull a model (e.g., llama)
print("Pulling the llama model...")
subprocess.run(['ollama', 'pull', model_name])

### On your local machine
Be sure that Ollama is runnung (follow the instructions in the prrevious lab). In the terminal, pull the `llama3.1:8b` model or another [tool model](https://ollama.com/search?c=tools). Start with small models. The code below has been tested with `llama3.1:8b`  (**llama3.1:8b is about 5GB**). Other models may require some adjustments in the prompt. Check the documentation of the model you want to use or test other models.

```bash
ollama run llama3.1:8b
```

---

## Teach your model to do a simple calculation (example)
*The example below have been adapted from this example on the [Ollama repository](https://github.com/ollama/ollama-python/blob/main/examples/tools.py)*. 

### Step 1: Import the necessary libraries

We use `ChatResponse` and `chat`.

In [2]:
from ollama import ChatResponse, chat

### Step 2: Define the functions (tools)

In this example, we define two functions: `add_two_numbers` and `subtract_two_numbers`. These functions will be used by the model to perform simple calculations.

Basically, each function defines a prompt for the model and what the model should do with the input. As you can see in the example, the prompte structure is does not need to be complex or always the same.

In [3]:
# We strictly define the types of the arguments and the return value to be sure that the tool will work as expected
def add_two_numbers(a: float, b: float) -> float:
  """
  Add two numbers

  Args:
    a (float): The first number
    b (float): The second number

  Returns:
    float: The sum of the two numbers
  """

  # The cast is necessary as returned tool call arguments don't always conform exactly to schema
  # E.g. this would prevent "what is 30 + 12" to produce '3012' instead of 42
  return float(a) + float(b)

def multiply_two_numbers(a: float, b: float) -> float:
  """
  Multiply two numbers

  Args:
    a (float): The first number
    b (float): The second number

  Returns:
    float: The multiplication of the two numbers
  """

  # The cast is necessary as returned tool call arguments don't always conform exactly to schema
  return float(a) * float(b)

def divide_two_numbers(a: float, b: float) -> float:
  """
  Divide two numbers

  Args:
    a (float): The first number
    b (float): The second number

  Returns:
    float: The division of the two numbers
  """

  # The cast is necessary as returned tool call arguments don't always conform exactly to schema
  return float(a) / float(b)


# NOTE: subtract_two_numbers function is manually defined in the cell below to show the "manual" way of defining a tool
def subtract_two_numbers(a: float, b: float) -> float:
  """
  Subtract two numbers
  """

  # The cast is necessary as returned tool call arguments don't always conform exactly to schema
  return float(a) - float(b)





In the examples above, the tools are defined as functions. However, tools can also be manually defined as dictionaries (see below).

In [4]:
# Tools can still be manually defined and passed into chat
subtract_two_numbers_tool = {
  'type': 'function',
  'function': {
    'name': 'subtract_two_numbers', # The name of the function will be used by the model to call the right function (defined above)
    'description': 'Subtract two numbers.',
    'parameters': {
      'type': 'object',
      'required': ['a', 'b'],
      'properties': {
        'a': {'type': 'float', 'description': 'The first number'},
        'b': {'type': 'float', 'description': 'The second number'},
      },
    },
  },
}

In [5]:
# We create a dictionary that maps the function names to the functions

available_functions = {
  'add_two_numbers': add_two_numbers,
  'multiply_two_numbers': multiply_two_numbers,
  'divide_two_numbers': divide_two_numbers,
  'subtract_two_numbers': subtract_two_numbers,
}

### Step 3: Define the chat with the models.
We define the message, we inform the model about the available tools, and we call the model.

In [ ]:
messages = [{'role': 'user', 'content': 'What is three minus one?'}]
print('Prompt:', messages[0]['content'])

response: ChatResponse = chat( # here the `:` say that response is of type ChatResponse
  model_name, # change this to the model you want to use
  messages=messages,
  tools=[add_two_numbers, multiply_two_numbers, divide_two_numbers, subtract_two_numbers_tool], # we indicate the tools available to the model
)

Prompt: Add 2 and 3. Then subtract 1 from the result.


In [16]:
print('Response:', response.message.tool_calls) # print this for debugging/undersanding the tool calls

Response: [ToolCall(function=Function(name='add_two_numbers', arguments={'a': 2, 'b': 3})), ToolCall(function=Function(name='subtract_two_numbers', arguments={'a': '5', 'b': 1}))]


**Questions**:
- The response contains the tool(s) to call with its arguments. Try to ask a question that is not related to the tools. What does the model do? 
- How does the model decide which tool to use?
- Try the following questions:
    - Add 2 and 3. Then subtract 1 from the result.
    - Devide 380 by 13. Then multiply the result by 5.


<details>
<summary><strong>Answers</strong>:</summary>
<br>

- The `response.message.tool_calls` list will be empty or propose non-existent tools.
- The model decides which tool to use based on the prompt, on one side, and on the other side, on the tool's prompt. The model will use the tool that is more related to the prompt. For instance, if you ask for a sum, it will use the `add_two_numbers` function. If you ask for a subtraction, it will use the `subtract_two_numbers` function. If you ask for a multiplication, it will not find any tool and will return an empty list.
- The model understands that it has to call two tools. Also the order is correct. However, the models does not have the result of the first tool to use it in the second one. The model will call the first tool and then call the second one with an hallicinated result of the first one. This is a limitation of the current implementation. There are tools that allows doing that (e.g., [LangGraph](https://langchain-ai.github.io/langgraph/tutorials/introduction/)). For now, we have to do it manually. 
</details>

In [17]:
# we check if the model returned any tool calls
if response.message.tool_calls:
  # There may be multiple tool calls in the response, we treat them one by one ina for loop
  for tool in response.message.tool_calls:
    # Ensure the function is available, and then call it
    
    function_to_call = available_functions.get(tool.function.name) # here we get the function to call from the dictionary
    if function_to_call:
      print('Calling function:', tool.function.name)
      print('Arguments:', tool.function.arguments)
      # The ** operator unpacks the dictionary into keyword arguments
      # In the example 'What is three minus one?', the instruction below will be equivalent to call 'add_two_numbers(a=3, b=1)'
      output = function_to_call(**tool.function.arguments)
      print('Function output:', output)
    else:
      print('Function', tool.function.name, 'not found')



Calling function: add_two_numbers
Arguments: {'a': 2, 'b': 3}
Function output: 5.0
Calling function: subtract_two_numbers
Arguments: {'a': '5', 'b': 1}
Function output: 4.0


### Step 5: The final "ribbon"	
We use the LLM once again to process the response and print the result in a nice way.

In [ ]:
# Only needed to chat with the model using the tool call results
if response.message.tool_calls:
  # Add the function response to messages for the model to use
  messages.append(response.message)
  messages.append({'role': 'tool', 'content': str(output), 'name': tool.function.name})

else: 
  print('No tool calls returned from model, just providing the response to the user...')
  # we do not need to modify the messages if the model did not return any tool calls

# Get final response from model with function outputs
final_response = chat(model_name, messages=messages)
print('Final response:', final_response.message.content)


## Challenge!

Now, it's up to you. 

Options:
1. Implement an agent that chooses whether to use information coming from its own knowledge base, from a vector store (RAG) or other ways (e.g., the [web](https://tavily.com/)). This approach of letting the model choosing the best "route" to get its information and provide the best answer is known as "routing".
2. Implement an agent that send commands to the Home IO automation devices.
    - The commands can control the garage door, the lights, the store, etc.
    - Optional: add a vocal interface (e.g., using "whisper")
    - To get started: see the `readme.md` in the homeIO folder.
3. Implement an agent that propose a planning based on the meteorological data (e.g., using the [OpenMeteo API](https://open-meteo.com/)). The agent should be able to suggest a plan for the nex day/week based on the weather forecast. For instance, if it is sunny, the agent could suggest going for a walk or having a picnic. If it is rainy, the agent could suggest indoor activities such as reading or watching a movie.
6. Implement an agent that retrieve data from a dataset (e.g., download a different dataset from kaggle) and perform simple clustering on the data
7. Propose your own idea!